# Herclassificering van LGN-kaart

In [2]:
import arcpy
import csv

def reclass_from_csv(input_raster, csv_path, output_raster):

    # ---------------------------------------------------------
    # FIX: Clear invalid snap raster
    # ---------------------------------------------------------
    arcpy.env.snapRaster = None

    # ---------------------------------------------------------
    # 1. Read CSV into dictionary
    # ---------------------------------------------------------
    reclass_dict = {}

    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            nummer = int(row["nummer"])
            reclass_value = int(row["reclass"])
            reclass_dict[nummer] = reclass_value

    # ---------------------------------------------------------
    # 2. Convert to RemapValue
    # ---------------------------------------------------------
    remap_list = [[k, v] for k, v in reclass_dict.items()]
    remap = arcpy.sa.RemapValue(remap_list)

    # ---------------------------------------------------------
    # 3. Reclassify
    # ---------------------------------------------------------
    out = arcpy.sa.Reclassify(input_raster, "Value", remap, "NODATA")

    # ---------------------------------------------------------
    # 4. Save output
    # ---------------------------------------------------------
    out.save(output_raster)

    print("Reclassification complete!")
    print(f"Output saved to: {output_raster}")

In [3]:
arcpy.env.workspace = r"D:\01_data\02_landgebruik\WSS"

input_raster = r"D:\01_data\02_landgebruik\WSS\Centraal Holland\lgn_func_CH.tif"
csv_path = r"D:\01_data\02_landgebruik\WSS\reclass_table_def.csv"
lu_reclass = r"D:\01_data\02_landgebruik\WSS\Centraal Holland\landgebruik_reclass_CH_def_v2.tif"

reclass_from_csv(input_raster, csv_path, lu_reclass)


Reclassification complete!
Output saved to: D:\01_data\02_landgebruik\WSS\Centraal Holland\landgebruik_reclass_CH_def_v2.tif


# Creeër LGN-kaart per rekengebied

In [107]:
import arcpy
import os
from tqdm import tqdm
import re

def maak_veilige_naam(
    naam,
    *,
    target="gdb_object",   # "gdb_object" of "filesystem"
    workspace=None,
    max_len=60
):
    """
    Maakt een veilige naam voor:
    - target="filesystem"  → mappen + .gdb namen
    - target="gdb_object"  → feature classes / tabellen in een GDB
    """

    s = str(naam).strip().lower()

    # Uniforme normalisatie
    s = s.replace(" ", "_")
    s = re.sub(r"[^\w]", "_", s)   # ook - / \ etc.
    s = re.sub(r"_+", "_", s)

    # Niet beginnen met cijfer
    if s and s[0].isdigit():
        s = f"p_{s}"

    if target == "filesystem":
        return s.strip("_")

    if target == "gdb_object":
        if workspace is None:
            raise ValueError("workspace is verplicht bij target='gdb_object'")

        s = s[:max_len]
        s = s.strip("_")
        return arcpy.ValidateTableName(s, workspace)

    raise ValueError(f"Onbekend target: {target}")
    
def clip_landuse_by_polder(
        input_raster,
        shapefile,
        root_folder,
        waterschap_field="waterschap",
        polder_field="polder",
        select_waterschap=None,
        select_polder=None,
        use_existing=None
    ):
    """
    Clip landuse raster per polder en sla op in bestaande
    polderstructuur.

    Parameters
    ----------
    input_raster : str
        Pad naar input raster.

    shapefile : str
        Shapefile of feature class met polders.

    root_folder : str
        Hoofdmap waarin de waterschap/polder structuur staat.

    waterschap_field : str
        Veldnaam met waterschap.

    polder_field : str
        Veldnaam met poldernaam.

    select_waterschap : str, optional
        Verwerk alleen dit waterschap.

    select_polder : str of list[str], optional
        Verwerk alleen deze polder(s).

    use_existing : bool, optional
        Indien True worden bestaande rasters overgeslagen.
    """

    arcpy.env.overwriteOutput = True

    # Alle features ophalen voor tqdm
    features = []
    with arcpy.da.SearchCursor(
        shapefile,
        ["SHAPE@", waterschap_field, polder_field]
    ) as cursor:
        for row in cursor:
            features.append(row)

    print(f"📦 {len(features)} polders gevonden")

    # Selectie waterschap
    ws_sel = None
    if select_waterschap is not None:
        ws_sel = maak_veilige_naam(
            select_waterschap,
            target="filesystem"
        )

    # Selectie polder(s)
    polder_sel = None
    if select_polder is not None:

        if isinstance(select_polder, str):
            select_polder = [select_polder]

        polder_sel = {
            maak_veilige_naam(
                polder,
                target="filesystem"
            )
            for polder in select_polder
        }

    for shape, ws, pol in tqdm(
        features,
        desc="Clipping polders",
        unit="polder"
    ):

        ws_clean = maak_veilige_naam(
            ws,
            target="filesystem"
        )

        # Filter waterschap
        if ws_sel and ws_clean != ws_sel:
            continue

        polder_clean = maak_veilige_naam(
            pol,
            target="filesystem"
        )

        # Filter polder(s)
        if polder_sel and polder_clean not in polder_sel:
            continue

        ws_pad = os.path.join(
            root_folder,
            ws_clean
        )

        polder_pad = os.path.join(
            ws_pad,
            polder_clean
        )

        # Alleen bestaande mappen gebruiken
        if not os.path.exists(polder_pad):
            print(f"⏭️ Overgeslagen (geen map): {polder_clean}")
            continue

        out_name = f"{ws_clean}_{polder_clean}_lu.tif"
        out_path = os.path.join(
            polder_pad,
            out_name
        )

        if use_existing and arcpy.Exists(out_path):
            print(f"⏭️ Bestaat al, overgeslagen: {out_name}")
            continue

        try:
            arcpy.management.Clip(
                in_raster=input_raster,
                rectangle="#",
                out_raster=out_path,
                in_template_dataset=shape,
                clipping_geometry="ClippingGeometry",
                maintain_clipping_extent="MAINTAIN_EXTENT"
            )

        except Exception as e:
            print(f"❌ Fout bij {pol}: {e}")

    print("✅ All polders processed successfully.")

In [108]:
lu_reclass = r"D:\01_data\02_landgebruik\WSS\Centraal Holland\landgebruik_reclass_CH_def_v2.tif"
root_folder = r"D:\04_results"
shapefile = r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\02_data_raw\Peilgebieden\Centraal Holland\afvoergebieden_compleet.shp"


clip_landuse_by_polder(
    input_raster=lu_reclass,
    shapefile=shapefile,
    root_folder=root_folder,
    waterschap_field="Waterschap",
    polder_field="Naam_1",
    select_waterschap="HDSR",
    select_polder=[
        "Houten",
    "De Pleyt"],
    use_existing=False
)


📦 518 polders gevonden


Clipping polders: 100%|██████████| 518/518 [01:09<00:00,  7.50polder/s]﻿


✅ All polders processed successfully.
